# Edit N-grams
Demo code for training an $N$-gram LM on string edits.
Uses `Levenshtein` package to convert a (query, target) pair of strings to a list of (intab, outtab) pairs.

In [3]:
from Levenshtein import editops

`Levenshtein.editops` returns a list of triples `('edit', i, j)` 

In [13]:
editops("foo", "boo")

[('replace', 0, 0)]

Convert the list of triples to a list of character pairs.

In [26]:
_epsilon = "<Empty>"


def get_edit_corpus(corpus: list[tuple[str, str]]) -> list[tuple[str, str]]:
    edit_pairs = []
    for in_word, out_word in corpus:
        edit_pairs.extend(get_edit_pairs(in_word, out_word))
    return edit_pairs


def get_edit_pairs(in_word: str, out_word: str) -> list[tuple[str, str]]:
    operations = editops(in_word, out_word)
    char_pairs = [
        operation_to_char_pair(operation, in_word, out_word) for operation in operations
    ]
    return char_pairs


def operation_to_char_pair(
    operation: tuple[str, int, int], in_word: str, out_word: str
) -> tuple[str, str]:
    operation_type, i, j = operation
    match operation_type:
        case "insert":
            return (_epsilon, out_word[j])
        case "replace":
            return (in_word[i], out_word[j])
        case "delete":
            return (in_word[i], _epsilon)


get_edit_pairs("food", "bfoobar")

[('<Empty>', 'b'), ('<Empty>', 'b'), ('<Empty>', 'a'), ('d', 'r')]

## Bigram LM estimation
Learn a bigram model to estimate $\mathrm{P}(c_j|c_i)$.

In [21]:
from nltk.lm import MLE
from string import ascii_lowercase

In [85]:
lm = MLE(2)

vocab = [_epsilon] + list(ascii_lowercase)
vocab[:5]

['<Empty>', 'a', 'b', 'c', 'd']

In [75]:
corpus = [
    ("foo", "fo"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("foo", "vaa"),
    ("bar", "bat"),
    ("baz", "pid"),
]

edit_corpus = get_edit_corpus(corpus)
edit_corpus[:5]

[('o', '<Empty>'), ('f', 'v'), ('o', 'a'), ('o', 'a'), ('f', 'v')]

In [76]:
lm.fit([edit_corpus], vocabulary_text=vocab)

In [78]:
lm.entropy([('o', 'a')])

0.2630344058337938

In [82]:
lm.score(context='o', word='a')

0

Don't need NLTK though.
We can just use a $|V\cup\emptyset|\times|V\cup\emptyset|$ transition matrix (as long as we're sticking to bigrams).

In [90]:
import numpy as np
from scipy.special import softmax

In [96]:
def get_random_transition_matrix(n):
    mat = np.random.random((n,n))
    probs = softmax(mat, axis=0)

    return probs

mat = get_random_transition_matrix(len(vocab))
mat[:,0].sum()

np.float64(0.9999999999999999)